# 02 · Atribuição de commits a agentes de IA

**Objetivo.** Identificar, no histórico de cada projeto do piloto, quais commits foram feitos por agentes de IA, combinando os PRs do AIDev com assinaturas textuais deixadas pelos agentes, e definir a data de adoção de cada projeto.

- **Lê:** `../data/pull_request.parquet`, `../data/pr_commits.parquet`, `../out/piloto.csv` e `../out/<projeto>/log.txt` (histórico extraído no passo 3.2, que nenhum notebook desta pasta gera)
- **Grava:** `../out/<projeto>/commits_atribuidos.csv` (um por projeto) e `../out/atribuicao_resumo.csv`

Nos caminhos, `<projeto>` é o nome do repositório com `/` trocado por `__`.

**O que esta célula faz.** Define as regras de atribuição e as funções que as aplicam ao histórico de cada projeto. Um commit é atribuído a um agente (i) se for merge ou squash de um PR aceito do AIDev, com data entre 1 dia antes da criação do PR e 2 dias depois do merge, (ii) se o SHA constar nos commits de um PR do AIDev (caso de rebase) ou (iii) se nenhuma das anteriores valer e autor, e-mail ou mensagem (inclusive o nome do branch nos merges) trouxerem a assinatura de um agente, só a partir de 2024-03-01. A função `resumo` também define a adoção como a data do primeiro commit de agente, verifica se há pelo menos 12 meses de histórico antes dela e 30 ou mais commits nesse período, e conta os casos em que AIDev e assinatura apontam agentes diferentes.

In [1]:
import os, re, warnings
import pandas as pd

warnings.filterwarnings('ignore', category=DeprecationWarning)   # aviso interno do pandas com o NumPy novo

ERA_AGENTES = pd.Timestamp('2024-03-01', tz='UTC')   # nenhum agente de codificação existia antes
ASSINATURAS = [   # 3.4: marcas que cada agente deixa em autor, e-mail, mensagem ou nome do branch do merge
    ('Copilot', re.compile(r'copilot-swe-agent|198982749\+Copilot@users\.noreply|\bfrom [^\s]+/copilot/', re.I)),
    ('Devin', re.compile(r'devin-ai-integration|\bfrom [^\s]+/devin/', re.I)),
    ('OpenAI_Codex', re.compile(r'chatgpt-codex-connector|\bfrom [^\s]+/codex[/-]', re.I)),
    ('Cursor', re.compile(r'cursoragent@cursor\.com|Co-authored-by: Cursor Agent|\bfrom [^\s]+/cursor/', re.I)),
    ('Claude_Code', re.compile(r'Generated with \[Claude Code\]|Claude <noreply@anthropic\.com>|\bfrom [^\s]+/claude/', re.I)),
    ('Jules', re.compile(r'google-labs-jules', re.I)),
]

pr = pd.read_parquet('../data/pull_request.parquet')
pc = pd.read_parquet('../data/pr_commits.parquet')


def ler_historico(chave, base='..'):
    """Lê o log.txt gerado no passo 3.2."""
    texto = open(f'{base}/out/{chave}/log.txt', encoding='utf-8', errors='replace').read()
    linhas = []
    for rec in texto.split('\x1e')[1:]:
        sha, pais, autor, email, data, msg = rec.split('\x1f', 5)
        linhas.append(dict(sha=sha, autor=autor, email=email, msg=msg,
                           data=pd.Timestamp(data).tz_convert('UTC')))
    return pd.DataFrame(linhas).sort_values('data').reset_index(drop=True)


def assinatura(c):
    """3.4: agente indicado pela assinatura do commit, ou None."""
    if c.data < ERA_AGENTES:
        return None
    blob = f'{c.autor} <{c.email}>\n{c.msg}'
    for agente, rx in ASSINATURAS:
        if rx.search(blob):
            return agente
    return None


def atribuir(nome, base='..'):
    chave = nome.replace('/', '__')
    h = ler_historico(chave, base)
    # PRs aceitos do AIDev neste repositório
    prs = pr[(pr.repo_url.str.lower() == f'https://api.github.com/repos/{nome}'.lower()) & pr.merged_at.notna()]
    num2agente = dict(zip(prs.number, prs.agent))
    # o commit no ramo principal deve cair entre 1 dia antes da criação do PR e 2 dias depois do merge
    criado = pd.to_datetime(prs.created_at, utc=True) - pd.Timedelta('1D')
    aceito = pd.to_datetime(prs.merged_at, utc=True) + pd.Timedelta('2D')
    janela = dict(zip(prs.number, zip(criado, aceito)))
    shas_pr = pc[pc.pr_id.isin(prs.id)].merge(prs[['id', 'agent']], left_on='pr_id', right_on='id')
    sha2agente = dict(zip(shas_pr.sha, shas_pr.agent))
    no_historico = set(h.sha)

    agente, fonte, assin = [], [], []
    for c in h.itertuples():
        a_sig = assinatura(c)
        assin.append(a_sig)
        titulo = c.msg.split('\n', 1)[0]
        m = re.search(r'Merge pull request #(\d+)', titulo) or re.search(r'\(#(\d+)\)\s*$', titulo)
        if m and int(m.group(1)) in num2agente and janela[int(m.group(1))][0] <= c.data <= janela[int(m.group(1))][1]:
            agente.append(num2agente[int(m.group(1))]); fonte.append('aidev')            # 3.3 merge ou squash
        elif c.sha in sha2agente and c.sha in no_historico:
            agente.append(sha2agente[c.sha]); fonte.append('aidev')                      # 3.3 rebase
        elif a_sig:
            agente.append(a_sig); fonte.append('assinatura')                             # 3.4
        else:
            agente.append(None); fonte.append(None)
    h['agente'], h['fonte'], h['assinatura'] = agente, fonte, assin
    h['is_agente'] = h.agente.notna()
    h['titulo'] = h.msg.str.split('\n').str[0]
    h.drop(columns='msg').to_csv(f'{base}/out/{chave}/commits_atribuidos.csv', index=False)
    return h, len(prs)


def resumo(nome, adocao_aidev, base='..'):
    h, n_prs = atribuir(nome, base)
    ag = h[h.is_agente]
    adocao = ag.data.min() if len(ag) else None
    pre12 = int(((h.data >= adocao - pd.DateOffset(months=12)) & (h.data < adocao)).sum()) if adocao is not None else 0
    inclui = bool(adocao is not None and h.data.min() <= adocao - pd.DateOffset(months=12) and pre12 >= 30)
    # 3.6 (verificações automáticas): onde a assinatura discorda do AIDev
    ai = h[h.fonte == 'aidev']
    discorda = int((ai.assinatura.notna() & (ai.assinatura != ai.agente)).sum())
    return dict(projeto=nome, commits=len(h), prs_aidev=n_prs,
                agente_aidev=int((h.fonte == 'aidev').sum()), agente_assinatura=int((h.fonte == 'assinatura').sum()),
                pct_agente=round(100 * len(ag) / len(h), 1),
                agente_principal=ag.agente.value_counts().idxmax() if len(ag) else None,
                adocao=adocao.strftime('%Y-%m-%d') if adocao is not None else None,
                mes_adocao_aidev=pd.Timestamp(adocao_aidev).strftime('%Y-%m'),
                commits_12m_antes=pre12, inclui=inclui, aidev_x_assinatura_discordam=discorda)

**O que esta célula faz.** Aplica `resumo` a cada projeto de `piloto.csv`, grava `../out/atribuicao_resumo.csv` e exibe a tabela com os commits atribuídos por fonte (AIDev ou assinatura), a data de adoção e as verificações automáticas. O processamento também grava o `commits_atribuidos.csv` de cada projeto.

In [2]:
piloto = pd.read_csv('../out/piloto.csv')
tabela = pd.DataFrame([resumo(p.full_name, p.primeiro) for _, p in piloto.iterrows()])
tabela.to_csv('../out/atribuicao_resumo.csv', index=False)
tabela

,projeto,commits,prs_aidev,agente_aidev,agente_assinatura,pct_agente,agente_principal,adocao,mes_adocao_aidev,commits_12m_antes,inclui,aidev_x_assinatura_discordam
0,numbagg/numbagg,791,12,12,28,5.1,Claude_Code,2025-06-17,2025-06,56,True,0
1,econ-ark/HARK,1063,8,8,7,1.4,OpenAI_Codex,2025-05-20,2025-05,71,True,0
2,NewFuture/DDNS,596,13,13,68,13.6,Copilot,2025-07-01,2025-07,115,True,0
3,lidofinance/diffyscan,193,6,6,0,3.1,OpenAI_Codex,2025-05-22,2025-05,36,True,0
4,airbytehq/PyAirbyte,588,14,9,234,41.3,Devin,2025-01-28,2025-01,246,True,0


**O que isto mostra.** Os históricos têm 791 (numbagg), 1063 (HARK), 596 (DDNS), 193 (diffyscan) e 588 (PyAirbyte) commits, dos quais 5.1%, 1.4%, 13.6%, 3.1% e 41.3% foram atribuídos a agentes; em PyAirbyte, 9 dos 14 PRs aceitos do AIDev foram localizados no histórico e 234 commits vêm só da assinatura, enquanto em diffyscan a atribuição é integralmente do AIDev (6 commits, 0 por assinatura). O percentual de PyAirbyte depende, portanto, da heurística de assinatura, que não foi validada manualmente, e os valores de HARK (7 commits por assinatura) e diffyscan se baseiam em poucos commits. A coluna `aidev_x_assinatura_discordam` é 0 nos cinco projetos, o que confirma que, nos commits atribuídos pelo AIDev que têm assinatura, esta nunca aponta para outro agente (a verificação não cobre commits sem assinatura). A adoção cai no mesmo mês de `mes_adocao_aidev` nos cinco projetos, e `inclui` é True em todos, com 36 (diffyscan) a 246 (PyAirbyte) commits nos 12 meses anteriores à adoção, acima do mínimo de 30.